In [1]:
#bypassing ssl certificate verification

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

#Imports

import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import contextlib
import matplotlib
matplotlib.use("Agg")         
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
import cvxpy as cp
from datetime import datetime
from scipy.optimize import linear_sum_assignment
from fredapi import Fred

In [2]:
#Configuring sources

#Loading FRED API
from dotenv import load_dotenv
load_dotenv()

FRED_API_KEY = os.environ.get("FRED_API_KEY")
if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY not set — create a .env file with FRED_API_KEY=your_key")  

ASSETS = {
    "SPY": "US Equities (S&P 500) — also the HMM's regime proxy index",
    "TLT": "Long-Term Treasuries",
    "GLD": "Gold",
    "EEM": "Emerging Markets",
    "VNQ": "Real Estate",
    "LQD": "Corp Bonds",
    "BIL": "3-Month T-Bill (cash proxy)",
    "QQQ": "Nasdaq  100",
}
SAFE_HAVEN = ["TLT", "GLD", "BIL", "LQD"]
CASH_PROXY = "BIL"
RISKY      = ["SPY", "EEM","VNQ","QQQ" ]  

START_DATE = "2010-01-01"
END_DATE   = "2026-5-31"         
N_REGIMES  = 2
REBAL_FREQ = "ME"                # Month End Rebalancing
TX_COST_BPS = 5


REGIME_LABELS = {1: "Bull",  -1: "Bear"}
REGIME_COLORS = {"Bull": "#4CAF50",  "Bear": "#F44336"}

# Oracle definition: Sign and magnitude of forward returns over a fixed horizon of 21 days
ORACLE_HORIZON        = 21  
ORACLE_THRESHOLD_MULT = 0.5  

'''
An Expanding window multi fold method is used for Walk Forward Validation. 
The training period expands as we move forward in time (first it is 7 years, then increases to 9, then 11 and so on). The test period is a fixed 2 year period.
A initial train period of 7 years is allowed for the model to get well acquainted before testing.
'''
WF_START            = "2010-01-01"   # fixed start of every fold's training window
INITIAL_TRAIN_YEARS = 7              
TEST_WINDOW_YEARS   = 2             
WF_END              = "2026-5-31"  

def _build_expanding_folds(start=WF_START, initial_train_years=INITIAL_TRAIN_YEARS,
                           test_window_years=TEST_WINDOW_YEARS, end=WF_END):
    start_ts = pd.Timestamp(start)
    end_ts   = pd.Timestamp(end)
    folds = []
    train_end = start_ts + pd.DateOffset(years=initial_train_years)
    i = 1
    while train_end < end_ts:
        test_start = train_end
        test_end   = min(train_end + pd.DateOffset(years=test_window_years), end_ts)
        folds.append({
            "name":        f"Fold {i}",
            "train_start": start_ts.strftime("%Y-%m-%d"),
            "train_end":   train_end.strftime("%Y-%m-%d"),
            "test_start":  test_start.strftime("%Y-%m-%d"),
            "test_end":    test_end.strftime("%Y-%m-%d"),
        })
        train_end = train_end + pd.DateOffset(years=test_window_years)
        i += 1
    return folds

FOLDS = _build_expanding_folds()

def print_fold_periods(folds, title="Fold time periods"):
    print(f"\n{title}")
    print(f"  {'Fold':10s} {'Train start':12s} {'Train end':12s} {'Test start':12s} {'Test end':12s}")
    for f in folds:
        print(f"  {f['name']:10s} {f['train_start']:12s} {f['train_end']:12s} "
              f"{f['test_start']:12s} {f['test_end']:12s}")

print_fold_periods(FOLDS, title="  Fold time periods used in backtest and final results\n")

FRED_SERIES = {
    "DGS3MO": "3-Month Treasury Bill Rate",
    "DGS10":  "10-Year Treasury Bond Rate",
}

os.makedirs("output", exist_ok=True)


  Fold time periods used in backtest and final results

  Fold       Train start  Train end    Test start   Test end    
  Fold 1     2010-01-01   2017-01-01   2017-01-01   2019-01-01  
  Fold 2     2010-01-01   2019-01-01   2019-01-01   2021-01-01  
  Fold 3     2010-01-01   2021-01-01   2021-01-01   2023-01-01  
  Fold 4     2010-01-01   2023-01-01   2023-01-01   2025-01-01  
  Fold 5     2010-01-01   2025-01-01   2025-01-01   2026-05-31  


In [3]:
#Importing Data

def download_prices(tickers, start, end):
    # Market prices from Yahoo Finance
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False, threads=False)
    if isinstance(raw.columns, pd.MultiIndex):
        close = raw["Close"][tickers]
        ohlc  = {t: raw.xs(t, axis=1, level=1)[["Open", "High", "Low", "Close"]] for t in tickers}
    else:
        close = raw[["Close"]].rename(columns={"Close": tickers[0]})
        ohlc  = {tickers[0]: raw[["Open", "High", "Low", "Close"]]}

    missing = close.columns[close.isna().all()].tolist()
    if missing:
        print(f"  \u26a0 Retrying failed tickers individually: {missing}")
        for t in missing:
            try:
                single = yf.download(t, start=start, end=end,
                                     auto_adjust=True, progress=False)
                close[t] = single["Close"]
                ohlc[t]  = single[["Open", "High", "Low", "Close"]]
            except Exception as e:
                print(f"    \u2717 {t} still failed: {e}")

    still_missing = close.columns[close.isna().all()].tolist()
    if still_missing:
        raise RuntimeError(f"Could not fetch price data for: {still_missing}.")

    close = close.dropna(how="all").ffill()
    return close, ohlc


def download_vix(start, end):
    # CBOE VIX from Yahoo Finance (^VIX)
    vix = yf.download("^VIX", start=start, end=end,
                      auto_adjust=True, progress=False)["Close"]
    vix.name = "VIX"
    print(f"  \u2713 CBOE VIX: {len(vix)} observations")
    return vix

def download_skew(start, end):
    # CBOE SKEW Index from Yahoo Finance (^SKEW) 
    skew = yf.download("^SKEW", start=start, end=end, progress=False)["Close"]
    return skew




def download_fred(api_key, series_dict, start, end):
    # Various Macro economic features from FRED
    try:
        from fredapi import Fred
        fred = Fred(api_key=api_key)
        frames = {}
        for sid, label in series_dict.items():
            try:
                s = fred.get_series(sid, observation_start=start, observation_end=end)
                s.name = sid
                frames[sid] = s
                print(f"  \u2713 FRED [{sid}]: {label}")
            except Exception as e:
                print(f"  \u2717 FRED [{sid}]: {e}")
        if frames:
            df = pd.DataFrame(frames)
            df.index = pd.to_datetime(df.index)
            return df.ffill()
        return pd.DataFrame()
    except ImportError:
        print("  \u26a0 fredapi not installed \u2014 skipping FRED  (pip install fredapi)")
        return pd.DataFrame()

In [4]:
def compute_returns(prices):
    return np.log(prices / prices.shift(1)).dropna()


def _ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def build_technical_features(spy_ohlc, spy_ret):
    high, low, close ,openy = spy_ohlc["High"], spy_ohlc["Low"], spy_ohlc["Close"],spy_ohlc["Open"]
    daily_range = ((high - low) / close).rename("daily_range")
    ema14_raw = _ema(close, 14)
    ema14     = (close / ema14_raw - 1.0).rename("ema14")
    macd_main = ((_ema(close, 12) - _ema(close, 26)) / close).rename("macd_main")
    ewma20_var  = (spy_ret ** 2).ewm(span=20, adjust=False).mean()
    ewma20      = np.sqrt(ewma20_var).rename("ewma20_vol")
    feat = pd.concat([  ema14, daily_range, ewma20, macd_main  ], axis=1)
    return feat

def build_macro_features(vix_series, fred_df, index, spy_ret, skew_series=None, spy_close=None):
    parts = {}
    have_vix = vix_series is not None and not vix_series.empty
    def to_1d(s):
        if hasattr(s, "ndim") and s.ndim == 2:
            s = s.iloc[:, 0]
        return s.squeeze()
    if have_vix:
        vix_aligned = vix_series.reindex(index).ffill()
        if hasattr(vix_aligned, "ndim") and vix_aligned.ndim == 2:
            vix_aligned = vix_aligned.iloc[:, 0]
        parts["vix"] = vix_aligned  
    if skew_series is not None and not skew_series.empty:
        skew_aligned = to_1d(skew_series).reindex(index).ffill()
        parts["skew_idx"] = skew_aligned.rename("skew_idx")
        parts["skew_idx_lag"] = skew_aligned.shift(1).rename("skew_idx_lag")
    if fred_df is not None and not fred_df.empty:
        if "DGS3MO" in fred_df.columns:
            t3mo = fred_df["DGS3MO"].reindex(index).ffill()
            parts["t3mo"] = t3mo
        if "DGS10" in fred_df.columns:
            t10y = fred_df["DGS10"].reindex(index).ffill()
            #parts["t10y"] = t10y
        if "DGS3MO" in fred_df.columns and "DGS10" in fred_df.columns:
            parts["yield_ratio"] = (fred_df["DGS3MO"] / fred_df["DGS10"]).reindex(index).ffill()

    return pd.DataFrame(parts)

def build_feature_matrix(spy_ohlc, spy_ret, vix_series, fred_df, skew_series=None):
    tech  = build_technical_features(spy_ohlc, spy_ret)
    macro = build_macro_features(vix_series, fred_df, spy_ret.index,spy_ret, skew_series=skew_series,  spy_close=spy_ohlc["Close"])
    feat  = pd.concat([tech, macro], axis=1).dropna()
    return feat

In [5]:
#HMM

def standardize_features(train_feat: pd.DataFrame, target_feat: pd.DataFrame):
    # Z-SCORE
    mu  = train_feat.mean()
    sig = train_feat.std().replace(0, 1.0)
    return (target_feat - mu) / sig, mu, sig

def fit_hmm(feat_df, n_components=N_REGIMES, n_iter=1000, n_restarts=8, base_seed=42):
    '''
    Fit a Gaussian HMM via Baum-Welch EM on the fold feature matrix. 
    The model is fit fresh every fold and then tested on that folds testing period.
    Since HMMs are sensitive to initialization, 8 random restarts are run per fold and the one with the 
    highest training log likelihood is used in the fold.
    '''
    best_model, best_score = None, -np.inf
    for k in range(n_restarts):
        model = GaussianHMM(
            n_components=n_components,
            covariance_type="diag",
            n_iter=n_iter,
            random_state=base_seed + k,
            tol=1e-3,
            init_params="stmc",
            params="stmc",
        )
        try:
            model.fit(feat_df.values)
            score = model.score(feat_df.values)
        except Exception:
            continue
        if score > best_score:
            best_model, best_score = model, score
    if best_model is None:
        raise RuntimeError("HMM failed to fit across all restarts")
    return best_model

def decode_regimes(model, feat_df):
    # Viterbi decoding : hmmlearn's .predict uses Viterbi by default
    states = model.predict(feat_df.values)   
    return pd.Series(states, index=feat_df.index, name="raw_state")

def build_oracle_labels(spy_close, index, horizon=ORACLE_HORIZON, threshold=0.02):
    '''
    2-state oracle labels : {-1, +1}. 
    Every day is classified based on the sign and magnitude of the forward horizon-day SPY return.
    The raw viterbi states cannot be labelled bear/bull directly thus we make use of the oracle labels.
    '''
    fwd_ret = spy_close.shift(-horizon) / spy_close - 1.0
    fwd_ret = fwd_ret.reindex(index)

    oracle = pd.Series(-1, index=index, dtype=int)   # default Bear
    oracle[fwd_ret >= threshold] = 1                # Bull
    return oracle, threshold

def align_states_to_oracle(raw_states, oracle):
    oracle_vals = np.array([-1, 1])   
    n_states    = raw_states.max() + 1
    C = np.zeros((n_states, len(oracle_vals)))

    common_idx = raw_states.index.intersection(oracle.index)
    rs = raw_states.loc[common_idx]
    ov = oracle.loc[common_idx]

    for j in range(n_states):
        for l, val in enumerate(oracle_vals):
            C[j, l] = np.sum((rs.values == j) & (ov.values == val))

    row_ind, col_ind = linear_sum_assignment(-C)
    mapping = {int(row): int(oracle_vals[col]) for row, col in zip(row_ind, col_ind)}

    for j in range(n_states):
        mapping.setdefault(j, 1)   
    return mapping


In [6]:
# Testing hmm accuracy

import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, cohen_kappa_score

LABELS_ORDER = ["Bull", "Bear"]

def _fold_confusion(regimes_sub, spy, REGIME_LABELS, threshold, horizon):
    fwd_ret = spy.shift(-horizon) / spy - 1.0
    fwd_ret = fwd_ret.reindex(regimes_sub.index)

    truth = pd.Series(-1, index=regimes_sub.index, dtype=int)
    truth[fwd_ret >= threshold] = 1
    truth_str = truth.map(REGIME_LABELS)
    hmm_str   = regimes_sub["regime"].map(REGIME_LABELS)

    aligned = pd.concat([hmm_str.rename("hmm"), truth_str.rename("truth")], axis=1, join="inner")
    aligned = aligned.loc[fwd_ret.reindex(aligned.index).notna()]
    if aligned.empty:
        return None

    cm = confusion_matrix(aligned["truth"], aligned["hmm"], labels=LABELS_ORDER)
    cm_df = pd.DataFrame(cm, index=LABELS_ORDER, columns=LABELS_ORDER)
    kappa = cohen_kappa_score(aligned["truth"], aligned["hmm"]) if len(aligned) > 1 else float("nan")
    return cm_df, kappa, aligned

def hmm_confusion_matrix(regimes_df, prices, REGIME_LABELS, fold_thresholds,
                         horizon=ORACLE_HORIZON):
    '''
    Check accuracy of the hmm.
    Cohen's Kappa: agreement between raters. Higher the Kappa, better the performance by the model.
    '''
    spy = prices["SPY"]
    if not spy.index.is_unique:
        spy = spy[~spy.index.duplicated(keep="last")].sort_index()

    fold_results = {}
    all_aligned  = []
    print(f"\n{'='*55}\nPER-FOLD (test dates only)\n{'='*55}")
    for fname in regimes_df["fold"].unique():
        sub = regimes_df[regimes_df["fold"] == fname]
        threshold = fold_thresholds.get(fname)
        if threshold is None:
            print(f"  ⚠ {fname}: no stored training threshold, skipping")
            continue

        out = _fold_confusion(sub, spy, REGIME_LABELS, threshold, horizon)
        if out is None:
            print(f"  ⚠ {fname}: no forward-return data available (tail of sample), skipping")
            continue
        cm_df, kappa, aligned = out
        n = len(aligned)
        fold_results[fname] = (cm_df, kappa, n)
        all_aligned.append(aligned)

        acc = np.trace(cm_df.values) / cm_df.values.sum()
        print(f"\n{fname}  (n={n}, test dates only)")
        print(cm_df.to_string())
        print(f"  Accuracy: {acc:.1%}   Kappa: {kappa:.3f}")

    if not all_aligned:
        raise RuntimeError("No fold had evaluable out-of-sample data.")

    combined = pd.concat(all_aligned)
    cm = confusion_matrix(combined["truth"], combined["hmm"], labels=LABELS_ORDER)
    cm_df = pd.DataFrame(cm, index=LABELS_ORDER, columns=LABELS_ORDER)
    kappa = cohen_kappa_score(combined["truth"], combined["hmm"])

    print(f"\n{'='*55}\nCOMBINED (all folds, test dates only, n={len(combined)})\n{'='*55}")
    print(cm_df.to_string())

    print("\nPer-state prediction accuracy (recall):")
    for lbl in LABELS_ORDER:
        n_true = cm_df.loc[lbl].sum()
        n_correct = cm_df.loc[lbl, lbl]
        acc = n_correct / n_true if n_true > 0 else float("nan")
        print(f"  {lbl:8s}: {acc:.1%}  ({n_correct}/{n_true})")

    overall_acc = np.trace(cm) / cm.sum()
    print(f"\nOverall accuracy: {overall_acc:.1%}")
    print(f"Cohen's Kappa: {kappa:.3f}  "
          f"(<0.2 slight | 0.2-0.4 fair | 0.4-0.6 moderate | 0.6-0.8 substantial | >0.8 strong)")

    return fold_results, cm_df, kappa, overall_acc

In [7]:
'''
─────────────────────────────────────────────────────────────
Convex portfolio optimisation — real cvxpy solve per rebalance
─────────────────────────────────────────────────────────────
Objective: 
  maximize   1. μ'w                          expected return
             2. λ · w'Σw                     variance penalty
             3. κ · ||w − anchor||²          pull toward regime target
             4. (c/1e4) · ||w − w_prev||₁    turnover / transaction cost

  subject to  1. sum(w) = 1
              2. w ≥ 0
              3. w ≤ w_max                     (per-asset cap)

─────────────────────────────────────────────────────────────
Regime specific variables: 
─────────────────────────────────────────────────────────────
  Bull:  λ small  → let μ dominate, concentrate freely
         w_max high
         anchor = BULL_WEIGHTS, κ small → loose pull

  Bear:  λ large  → variance-averse
         w_max low on risky names
         anchor = BEAR_WEIGHTS, κ large → tight pull to safety
─────────────────────────────────────────────────────────────
Basic Rules followed: 
─────────────────────────────────────────────────────────────
    - Higher allocation to assets with strong trailing returns.
    - Don't stray too far from the default allocation for the current regime.
    - Avoid high risk - high volatility is one of the important factors to account for allocation. Co-variance between assets is also taken into consideration.
    - Avoids excessive rebalancing.

'''

BULL_WEIGHTS = {
    "SPY": 0.35, "EEM": 0.005, "VNQ": 0.005, "GLD": 0.29,
    "TLT": 0.00, "BIL": 0.00, "LQD": 0.00, "QQQ": 0.35,
}

BEAR_WEIGHTS = {
    "SPY": 0.175, "EEM": 0.0, "VNQ": 0.0, "GLD": 0.60,
    "TLT": 0.01, "BIL": 0.01, "LQD": 0.01, "QQQ": 0.175,
}

# Per asset regime specific upper caps  
BULL_CAPS = {"SPY": 0.45, "EEM": 0.15, "VNQ": 0.15, "GLD": 0.55,
             "TLT": 0.20, "BIL": 0.20, "LQD": 0.20, "QQQ": 0.45}

BEAR_CAPS = {"SPY": 0.30, "EEM": 0.05, "VNQ": 0.05, "GLD": 0.60,
             "TLT": 0.15, "BIL": 0.15, "LQD": 0.15, "QQQ": 0.35}

# Risk-aversion and anchor strength per regime
LAMBDA_RISK = {1: 0.20,  -1: 3.00}   # Bull: chase returns. Bear: play it safe
KAPPA_ANCHOR = {1: 2.0,  -1: 6.0}    # Bear pulled harder to its safer anchor
MU_SHRINK   = 0.5                 # shrink sample mean toward 0 (mu is noisy)


def optimise_portfolio(returns_window: pd.DataFrame, regime: int,
                        tx_cost_bps: int = TX_COST_BPS,
                        prev_weights: np.ndarray = None) -> np.ndarray:
    tickers = list(returns_window.columns)
    n = len(tickers)

    weights_map = BULL_WEIGHTS if regime == 1 else BEAR_WEIGHTS
    caps_map    = BULL_CAPS    if regime == 1 else BEAR_CAPS

    anchor = np.array([weights_map.get(t, 0.0) for t in tickers])
    anchor = anchor / anchor.sum() if anchor.sum() > 0 else np.ones(n) / n
    caps   = np.array([caps_map.get(t, 1.0) for t in tickers])

    mu_sample = returns_window.mean().values * 252
    mu = (1 - MU_SHRINK) * mu_sample + MU_SHRINK * 0.0  

    Sigma_sample = returns_window.cov().values * 252
    Sigma = Sigma_sample + 1e-4 * np.eye(n)             

    lam   = LAMBDA_RISK[regime]
    kappa = KAPPA_ANCHOR[regime]

    prev_w = prev_weights if prev_weights is not None else anchor

    w = cp.Variable(n)

    ret_term     = mu @ w
    risk_term    = cp.quad_form(w, cp.psd_wrap(Sigma))
    anchor_term  = cp.sum_squares(w - anchor)
    tcost_term   = cp.norm1(w - prev_w) * (tx_cost_bps / 1e4)

    objective = cp.Maximize(ret_term
                             - lam   * risk_term
                             - kappa * anchor_term
                             - tcost_term)

    constraints = [cp.sum(w) == 1, w >= 0, w <= caps]

    prob = cp.Problem(objective, constraints)
    try:
        prob.solve(solver=cp.ECOS)
        if w.value is None or prob.status not in ("optimal", "optimal_inaccurate"):
            raise RuntimeError(prob.status)
        w_opt = np.clip(w.value, 0, None)
        w_opt = w_opt / w_opt.sum()
    except Exception:
        # Fail-safe: fall back to the static anchor if the solver fails
        w_opt = anchor

    return w_opt

In [8]:
'''
Walk-forward backtest 

For each fold:
  1. Calculate z-score mean/std using only the training window
  2. Fit the HMM once on this normalized training data
  3. Using only training data, decode HMM states and match each state to a regime (bull/bear) 
  via the Hungarian algorithm, then lock this mapping for the whole fold
  4. Each month, re-run the frozen HMM on all data up to that date (standardized with training-only stats), 
  tag the current state with the frozen regime mapping, and rebalance weights using the trailing year of returns.
  5. Apply transaction cost drag on the rebalance day and then move to the next rebalance day
'''
def walk_forward_backtest_paper(feat_df, rets, spy_close, tx_cost_bps=TX_COST_BPS):
    pnl_list, weight_rows, regime_rows = [], [], []
    fold_models, fold_mappings, fold_thresholds = {}, {}, {}
    prev_w = None

    for fold in FOLDS:
        fname   = fold["name"]
        tr_lo   = pd.Timestamp(fold["train_start"])
        tr_hi   = pd.Timestamp(fold["train_end"])
        te_lo   = pd.Timestamp(fold["test_start"])
        te_hi   = pd.Timestamp(fold["test_end"])

        train_feat_raw = feat_df[(feat_df.index >= tr_lo) & (feat_df.index < tr_hi)]
        if len(train_feat_raw) < 100:
            print(f"  \u26a0 {fname}: insufficient training data, skipping")
            continue

        train_feat_z, fold_mu, fold_sig = standardize_features(train_feat_raw, train_feat_raw)

        model = fit_hmm(train_feat_z)                         

        oracle_train, threshold = build_oracle_labels(spy_close, train_feat_z.index)
        raw_train = decode_regimes(model, train_feat_z)
        mapping = align_states_to_oracle(raw_train, oracle_train)

        fold_models[fname]      = model
        fold_mappings[fname]    = mapping
        fold_thresholds[fname]  = threshold
        print(f"\n  {fname}  |  TRAIN: {tr_lo.date()} \u2192 {tr_hi.date()}  "
              f"TEST: {te_lo.date()} \u2192 {te_hi.date()}")
        print(f"    state\u2192regime map: { {k: REGIME_LABELS[v] for k, v in mapping.items()} }  "
)

        test_rets = rets[(rets.index >= te_lo) & (rets.index < te_hi)]
        rebal_dates = test_rets.resample(REBAL_FREQ).last().index
        rebal_dates = rebal_dates[(rebal_dates >= te_lo) & (rebal_dates < te_hi)]

        for i, rd in enumerate(rebal_dates):
            decode_feat_raw = feat_df[(feat_df.index >= tr_lo) & (feat_df.index <= rd)]
            if len(decode_feat_raw) < 60:
                continue
            decode_feat_z = (decode_feat_raw - fold_mu) / fold_sig
            raw_states = decode_regimes(model, decode_feat_z)
            regime = mapping[int(raw_states.iloc[-1])]

            train_window = rets[rets.index <= rd].iloc[-252:]
            if len(train_window) < 20:
                continue
            w = optimise_portfolio(train_window, regime, tx_cost_bps, prev_w)

            next_rd     = rebal_dates[i + 1] if i + 1 < len(rebal_dates) else te_hi
            period_rets = rets[(rets.index >= rd) & (rets.index < next_rd)]
            period_rets = period_rets[period_rets.index < te_hi]
            if period_rets.empty:
                continue

            drag = 0.0
            if prev_w is not None:
                drag = np.sum(np.abs(w - prev_w)) * tx_cost_bps / 10_000

            daily_pnl = period_rets @ w
            daily_pnl.iloc[0] = daily_pnl.iloc[0] - drag

            pnl_list.append(daily_pnl)
            weight_rows.append({"date": rd, "regime": regime, "fold": fname,
                                 **dict(zip(rets.columns, w))})
            regime_rows.append({"date": rd, "regime": regime, "fold": fname})
            prev_w = w

            print(f"  {rd.date()}  [{REGIME_LABELS[regime]:7s}]  "
                  f"{dict(zip(rets.columns, np.round(w, 2)))}")
        
        print(fname, "state counts in training window:", raw_states.value_counts().to_dict())
        print(fname, "transmat diag:", np.diag(model.transmat_)) # this should not be 1.
    
    port_ret = pd.concat(pnl_list).sort_index()
    port_ret = port_ret[~port_ret.index.duplicated(keep="last")]
    weights_df = pd.DataFrame(weight_rows).set_index("date")
    regimes_df = pd.DataFrame(regime_rows).set_index("date")
    return port_ret, weights_df, regimes_df, fold_models, fold_mappings, fold_thresholds

In [9]:
#Benchmark

def build_benchmarks(rets):
    '''60/40 (SPY/TLT) and equal-weight benchmarks.'''
    tickers = list(rets.columns)
    n = len(tickers)

    w60 = np.zeros(n)
    if "SPY" in tickers: w60[tickers.index("SPY")] = 0.30
    if "QQQ" in tickers: w60[tickers.index("QQQ")] = 0.30
    if "GLD" in tickers: w60[tickers.index("GLD")] = 0.25
    if "TLT" in tickers: w60[tickers.index("TLT")] = 0.15
    if w60.sum() == 0: w60 = np.ones(n) / n
    else: w60 /= w60.sum()

    return (rets @ w60).rename("60/40"), (rets @ (np.ones(n)/n)).rename("Equal Weight")

In [10]:
#Performance metrics

def metrics(ret_s: pd.Series, rf=0.0) -> dict:
    ann_ret = ret_s.mean() * 252
    ann_vol = ret_s.std()  * np.sqrt(252)
    sharpe  = (ann_ret - rf) / ann_vol if ann_vol else float("nan")

    down    = ret_s[ret_s < 0].std() * np.sqrt(252)
    sortino = (ann_ret - rf) / down if down else float("nan")

    cum     = (1 + ret_s).cumprod()
    max_dd  = ((cum / cum.cummax()) - 1).min()
    calmar  = ann_ret / abs(max_dd) if max_dd else float("nan")

    return {
        "Ann. Return (%)":  round(ann_ret * 100, 2),
        "Ann. Vol (%)":     round(ann_vol * 100, 2),
        "Sharpe":           round(sharpe,  3),
        "Sortino":          round(sortino, 3),
        "Max Drawdown (%)": round(max_dd  * 100, 2),
        "Calmar":           round(calmar,  3),
        "Total Return (%)": round((cum.iloc[-1] - 1) * 100, 2),
    }


def tearsheet(strat, b6040, beq):
    return pd.DataFrame({
        "Regime-Shift": metrics(strat),
        "60/40":        metrics(b6040),
        "EqWt":         metrics(beq),
    }).T


def tearsheet_by_fold(port_ret, bench_6040, bench_eq, regimes_df):
    '''Separate tear sheet per fold (paper reports Fold 1 / Fold 2 test performance).'''
    out = {}
    for fname in regimes_df["fold"].unique():
        idx = regimes_df.index[regimes_df["fold"] == fname]
        idx = idx.intersection(port_ret.index)
        if len(idx) == 0:
            continue
        seg = port_ret.loc[idx.min():idx.max()]
        b60 = bench_6040.reindex(seg.index).fillna(0)
        beq = bench_eq.reindex(seg.index).fillna(0)
        out[fname] = tearsheet(seg, b60, beq)
    return out


def compute_turnover_stats(weights_df, tx_cost_bps=TX_COST_BPS):
    '''
    weights_df has ticker columns + 'regime' + 'fold'. Turnover is the one-way
    L1 change in asset weights between consecutive rebalances (chronological,
    across fold boundaries too — the backtest carries prev_w through folds).
    '''
    asset_cols = [c for c in weights_df.columns if c not in ("regime", "fold")]
    w = weights_df[asset_cols].sort_index()
    turnover_series = w.diff().abs().sum(axis=1).fillna(0)

    n_years = (w.index[-1] - w.index[0]).days / 365.25
    avg_annual_turnover_pct = 100 * turnover_series.sum() / max(n_years, 1e-9)
    total_cost_drag_pct = 100 * (turnover_series * tx_cost_bps / 10_000).sum()

    return turnover_series, avg_annual_turnover_pct, total_cost_drag_pct


def regime_transition_stats(regimes_df):
    regimes = regimes_df["regime"]
    run_id = (regimes != regimes.shift(1)).cumsum()
    run_id.name = "run_id"        
    runs = regimes.groupby(run_id).agg(["first", "size"])
    runs.columns = ["regime", "n_rebalances"]
    n_switches = max(len(runs) - 1, 0)
    avg_run_len = runs["n_rebalances"].mean()
    by_regime = runs.groupby("regime")["n_rebalances"].mean().to_dict()
    by_regime = {REGIME_LABELS.get(k, k): round(v, 1) for k, v in by_regime.items()}
    return n_switches, round(avg_run_len, 1), by_regime


def benchmark_correlation(port_ret, bench_6040, bench_eq):
    idx = port_ret.index
    c6040 = port_ret.corr(bench_6040.reindex(idx).fillna(0))
    ceq   = port_ret.corr(bench_eq.reindex(idx).fillna(0))
    return c6040, ceq


def verdict_line(fold_tearsheets, fold_periods_by_name):
    wins, losses, worst_fold, worst_gap = [], [], None, 0.0
    for fname, fts in fold_tearsheets.items():
        gap = fts.loc["Regime-Shift", "Total Return (%)"] - fts.loc["60/40", "Total Return (%)"]
        (wins if gap >= 0 else losses).append(fname)
        if gap < worst_gap:
            worst_gap, worst_fold = gap, fname
    n = len(wins) + len(losses)
    line = f"Regime-Shift beats 60/40 in {len(wins)}/{n} folds"
    if worst_fold:
        fp = fold_periods_by_name.get(worst_fold, {})
        line += (f"; underperforms in {worst_fold} "
                 f"({fp.get('test_start')}\u2013{fp.get('test_end')}) by {abs(worst_gap):.2f}pp")
    return line

In [11]:
# Visualization

DARK_BG = "#0d0d0d"
PANEL   = "#111111"
TEXT_C  = "#e8e8e8"
ACCENT  = "#BFFF00"
GRID_C  = "#2a2a2a"

plt.rcParams.update({
    "text.color": TEXT_C, "axes.labelcolor": TEXT_C,
    "xtick.color": TEXT_C, "ytick.color": TEXT_C,
    "figure.facecolor": DARK_BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRID_C, "grid.color": GRID_C,
})


def _shade_regimes(ax, regimes_df):
    for _, row in regimes_df.iterrows():
        name = REGIME_LABELS.get(int(row["regime"]), "Neutral")
        ax.axvspan(row.name, row.name + pd.DateOffset(months=1),
                   alpha=0.08, color=REGIME_COLORS[name], linewidth=0)


def chart_equity_curves(port_ret, b6040, beq, regimes_df):
    fig, ax = plt.subplots(figsize=(14, 5))
    for s, lbl, col, ls in [
        (port_ret, "Regime-Shift", ACCENT,     "-"),
        (b6040,    "60/40",        "#4fc3f7",  "--"),
        (beq,      "Equal Weight", "#ff8a65",  ":"),
    ]:
        curet = (1 + s.reindex(port_ret.index).fillna(0)).cumprod()
        pct = (curet - 1) * 100
        ax.plot(pct, color=col, lw=2 if ls=="-" else 1.5, ls=ls, label=lbl)
    _shade_regimes(ax, regimes_df)
    patches = [mpatches.Patch(color=REGIME_COLORS[l], label=l) for l in REGIME_COLORS]
    h, l   = ax.get_legend_handles_labels()
    ax.legend(handles=h + patches, framealpha=0.25, labelcolor=TEXT_C, ncol=3)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_title("Cumulative % Returns over the years", color=ACCENT, fontsize=12)
    ax.grid(linewidth=0.4)
    fig.tight_layout()
    fig.savefig("output/01_cumulative_returns.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/01_cumulative_returns.png")


def chart_drawdown(port_ret, b6040, beq):
    fig, ax = plt.subplots(figsize=(14, 4))
    for s, lbl, col, alpha in [
        (port_ret, "Regime-Shift", ACCENT,    0.7),
        (b6040,    "60/40",        "#4fc3f7", 0.4),
        (beq,      "Equal Weight", "#ff8a65", 0.4),
    ]:
        cum = (1 + s).cumprod()
        dd  = (cum / cum.cummax()) - 1
        ax.fill_between(dd.index, dd.values, 0, alpha=1.0, facecolor="none", edgecolor=col, linewidth=1.2, label=lbl)
    ax.set_title("Drawdown Comparison", color=ACCENT, fontsize=12)
    ax.legend(framealpha=0.25, labelcolor=TEXT_C)
    ax.grid(linewidth=0.4)
    fig.tight_layout()
    fig.savefig("output/02_drawdown.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/02_drawdown.png")



def chart_rolling_sharpe(port_ret, b6040, beq, window=63):
    fig, ax = plt.subplots(figsize=(14, 4))
    for s, lbl, col in [
        (port_ret, "Regime-Shift", ACCENT),
        (b6040,    "60/40",        "#4fc3f7"),
        (beq,      "Equal Weight", "#ff8a65"),
    ]:
        rs = s.rolling(window).mean() / s.rolling(window).std() * np.sqrt(252)
        ax.plot(rs, color=col, lw=1.2, label=lbl)
    ax.axhline(0, color="#666", lw=0.8, ls="--")
    ax.set_title(f"Rolling {window}-Day Sharpe Ratio", color=ACCENT, fontsize=12)
    ax.legend(framealpha=0.25, labelcolor=TEXT_C)
    ax.grid(linewidth=0.4)
    fig.tight_layout()
    fig.savefig("output/03_rolling_sharpe.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/03_rolling_sharpe.png")


def chart_weight_allocation(weights_df):
    asset_cols = [c for c in weights_df.columns if c not in ("regime", "fold")]
    fig, ax = plt.subplots(figsize=(14, 5))
    weights_df[asset_cols].plot(kind="area", stacked=True, colormap="tab10",
                                linewidth=0, alpha=0.85, ax=ax)
    ax.set_title("Portfolio Weight Allocation Over Time", color=ACCENT, fontsize=12)
    ax.set_ylim(0, 1)
    ax.legend(framealpha=0.25, labelcolor=TEXT_C, loc="upper right")
    ax.grid(linewidth=0.4, axis="y")
    fig.tight_layout()
    fig.savefig("output/04_weight_allocation.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/04_weight_allocation.png")


def chart_regime_frequency(regimes_df):
    fig, ax = plt.subplots(figsize=(7, 5))
    counts = regimes_df["regime"].value_counts().sort_index()
    labels = [REGIME_LABELS.get(int(k), str(k)) for k in counts.index]
    colors = [REGIME_COLORS.get(l, "#888") for l in labels]
    bars   = ax.bar(labels, counts.values, color=colors, edgecolor=GRID_C, width=0.5)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(v), ha="center", color=TEXT_C, fontsize=11)
    ax.set_title("Regime Frequency (# rebalance months)", color=ACCENT, fontsize=12)
    ax.grid(linewidth=0.4, axis="y")
    fig.tight_layout()
    fig.savefig("output/05_regime_frequency.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/05_regime_frequency.png")


def chart_monthly_heatmap(port_ret):
    mr    = port_ret.resample("ME").sum()
    df    = mr.to_frame("ret")
    df["year"]  = df.index.year
    df["month"] = df.index.month
    pivot = df.pivot(index="year", columns="month", values="ret")
    pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                     "Jul","Aug","Sep","Oct","Nov","Dec"]
    vmax  = max(abs(pivot.values[~np.isnan(pivot.values)]).max(), 0.01)
    cmap  = mcolors.LinearSegmentedColormap.from_list(
                "rg", ["#F44336", "#111", "#4CAF50"])

    fig, ax = plt.subplots(figsize=(14, max(4, len(pivot)*0.45)))
    im = ax.imshow(pivot.values, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(12)); ax.set_xticklabels(pivot.columns, color=TEXT_C)
    ax.set_yticks(range(len(pivot))); ax.set_yticklabels(pivot.index, color=TEXT_C)
    for i in range(len(pivot)):
        for j in range(12):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v*100:.1f}%", ha="center", va="center",
                        fontsize=6.5, color="white" if abs(v) > vmax*0.4 else TEXT_C)
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    ax.set_title("Monthly Returns Heatmap (%)", color=ACCENT, fontsize=12)
    fig.tight_layout()
    fig.savefig("output/06_monthly_heatmap.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/06_monthly_heatmap.png")


def chart_hmm_transitions(model, fold_name, mapping=None):
    '''
    Dynamically sized to whatever model.transmat_ actually is (2x2 for the
    current 2-state Bull/Bear setup), instead of a hard-coded 3-label list.
    If `mapping` (raw state -> {-1, 1}) is supplied, axis labels show the
    actual Bull/Bear assignment for that fold instead of raw "S0"/"S1".
    '''
    tm = model.transmat_
    n_states = tm.shape[0]

    if mapping is not None:
        labels = [REGIME_LABELS.get(mapping.get(i, i), f"S{i}") for i in range(n_states)]
    else:
        labels = [f"S{i}" for i in range(n_states)]

    cmap   = mcolors.LinearSegmentedColormap.from_list("blues", ["#111", "#4fc3f7"])
    fig, ax = plt.subplots(figsize=(6, 5))
    im  = ax.imshow(tm, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(n_states)); ax.set_xticklabels(labels, color=TEXT_C)
    ax.set_yticks(range(n_states)); ax.set_yticklabels(labels, color=TEXT_C)
    ax.set_xlabel("To →", color=TEXT_C); ax.set_ylabel("From →", color=TEXT_C)
    for i in range(n_states):
        for j in range(n_states):
            ax.text(j, i, f"{tm[i, j]:.2f}", ha="center", va="center",
                    color="black" if tm[i, j] > 0.5 else TEXT_C, fontsize=10)
    ax.set_title(f"HMM Transition Matrix — {fold_name}", color=ACCENT, fontsize=11)
    fig.tight_layout()
    fname = f"output/07_hmm_transitions_{fold_name.replace(' ', '_')}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print(f"  Saved: {fname}")

def chart_hmm_transitions_grid(fold_models, fold_mappings):
    '''All fold transition matrices in one figure, one file.'''
    fnames = list(fold_models.keys())
    n = len(fnames)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()
    cmap = mcolors.LinearSegmentedColormap.from_list("blues", ["#111", "#4fc3f7"])

    for ax, fname in zip(axes, fnames):
        model  = fold_models[fname]
        tm     = model.transmat_
        n_states = tm.shape[0]
        mapping = fold_mappings.get(fname)
        if mapping is not None:
            labels = [REGIME_LABELS.get(mapping.get(i, i), f"S{i}") for i in range(n_states)]
        else:
            labels = [f"S{i}" for i in range(n_states)]

        ax.imshow(tm, cmap=cmap, vmin=0, vmax=1)
        ax.set_xticks(range(n_states)); ax.set_xticklabels(labels, color=TEXT_C, fontsize=9)
        ax.set_yticks(range(n_states)); ax.set_yticklabels(labels, color=TEXT_C, fontsize=9)
        for i in range(n_states):
            for j in range(n_states):
                ax.text(j, i, f"{tm[i, j]:.2f}", ha="center", va="center",
                        color="black" if tm[i, j] > 0.5 else TEXT_C, fontsize=9)
        ax.set_title(fname, color=ACCENT, fontsize=10)

    for ax in axes[n:]:
        ax.axis("off")  

    fig.suptitle("HMM Transition Matrices — All Folds", color=ACCENT, fontsize=13)
    fig.tight_layout()
    fig.savefig("output/07_hmm_transitions_all_folds.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print("  Saved: output/07_hmm_transitions_all_folds.png")

In [12]:
# Tearsheet

def build_combined_tearsheet(fold_tearsheets, fold_periods_by_name):
    '''Stack all per-fold tearsheets into one long-format DataFrame.'''
    rows = []
    for fname, fts in fold_tearsheets.items():
        fp = fold_periods_by_name.get(fname, {})
        fts_reset = fts.reset_index().rename(columns={"index": "Strategy"})
        fts_reset.insert(0, "Fold", fname)
        fts_reset.insert(1, "Test Start", fp.get("test_start"))
        fts_reset.insert(2, "Test End",   fp.get("test_end"))
        rows.append(fts_reset)
    combined = pd.concat(rows, ignore_index=True)
    combined.to_csv("output/tearsheet_all_folds.csv", index=False)
    print("Saved: output/tearsheet_all_folds.csv")
    return combined



In [13]:
#Main Pipeline

def run(fred_api_key=FRED_API_KEY):
    SEP = "\u2500" * 55

    # 1. Data Ingestion
    print(f"\n{SEP}\n\U0001F4E5  DATA INGESTION\n{SEP}")
    tickers = list(ASSETS.keys())

    # Source 1: Yahoo Finance — asset prices 
    close_px, ohlc = download_prices(tickers, START_DATE, END_DATE)
    print(f"  \u2713 Yahoo Finance prices: {close_px.shape[0]} days \u00d7 {close_px.shape[1]} assets")

    # Source 2: CBOE VIX — via Yahoo Finance ^VIX
    vix = download_vix(START_DATE, END_DATE)
    skew = download_skew(START_DATE, END_DATE)
    # Source 3: FRED API — macro economic indicators 
    fred_df = download_fred(fred_api_key, FRED_SERIES, START_DATE, END_DATE)
    if fred_df.empty:
        print("  \u26a0 FRED unavailable \u2014 yield-based features will be dropped")

    # 2. Feature Engineering 
    print(f"\n{SEP}\n\U0001F4D0  FEATURE ENGINEERING \n{SEP}")
    rets      = compute_returns(close_px)
    spy_ret   = np.log(ohlc["SPY"]["Close"] / ohlc["SPY"]["Close"].shift(1)).dropna()
    feat_df   = build_feature_matrix(ohlc["SPY"], spy_ret, vix, fred_df, skew_series=skew)
    rets_aligned = rets.reindex(feat_df.index).dropna()
    feat_df   = feat_df.reindex(rets_aligned.index)
    print(f"  Feature matrix: {feat_df.shape}")
    print(f"  Features: {list(feat_df.columns)}")

    print("  Correlation between features: ")
    feat_check = build_feature_matrix(ohlc["SPY"], spy_ret, vix, fred_df, skew_series=skew)
    display(feat_check.corr().round(2))

    # 3. Walk-Forward Backtest 
    print(f"\n{SEP}\n\U0001F501  WALK-FORWARD BACKTEST \n{SEP}")
    with open("output/walk_forward_backtest.txt", "w", encoding="utf-8") as f, contextlib.redirect_stdout(f):
        print(f"\n{SEP}\n\U0001F501  WALK-FORWARD BACKTEST \n{SEP}")
        port_ret, weights_df, regimes_df, fold_models, fold_mappings, fold_thresholds  = walk_forward_backtest_paper(
            feat_df, rets_aligned, close_px["SPY"]
        )
        print(f"\nBacktest range : {port_ret.index[0].date()} \u2192 {port_ret.index[-1].date()}")
        print(f"Rebalances     : {len(weights_df)}")
    

    print_fold_periods(FOLDS, title="Fold time periods used in walk-forward backtester and in results: ")

    print("\nRefer to walk_forward_backtest.txt for detailed walk forward result which includes details about hmm prediction and portfolio reallocation")

    
    with open("output/performance_tear_sheet.txt", "w", encoding="utf-8") as f, contextlib.redirect_stdout(f):

        # 4. Benchmarks + Tearsheet 
        SEP = "─" * 60          
        print(f"\n{SEP}\n             📊  PERFORMANCE TEAR SHEET\n{SEP}")
        r_idx = port_ret.index
        bench_6040, bench_eq = build_benchmarks(rets_aligned.reindex(r_idx).fillna(0))

        print(f"\n{SEP}\n1. Predictions by HMM Model \n{SEP}")
        fold_cm_results, cm_df, kappa, overall_acc = hmm_confusion_matrix(
            regimes_df, close_px, REGIME_LABELS, fold_thresholds
        )

        # Per-fold tear sheets
        print(f"\n{SEP}\n2. PER-FOLD TEAR SHEETS\n{SEP}")
        fold_periods_by_name = {f["name"]: f for f in FOLDS}
        fold_tearsheets = tearsheet_by_fold(port_ret, bench_6040, bench_eq, regimes_df)

        for fname, fts in fold_tearsheets.items():
            fp = fold_periods_by_name.get(fname)
            if fp:
                print(f"\n  {fname}  |  TRAIN: {fp['train_start']} → {fp['train_end']}   "
                    f"TEST: {fp['test_start']} → {fp['test_end']}")
            else:
                print(f"\n  {fname}:")
            print(fts.to_string(float_format=lambda x: f"{x:>10.2f}"))
        print()  
        combined_ts = build_combined_tearsheet(fold_tearsheets, fold_periods_by_name)

        print(f"\n{SEP}\n3. TEAR SHEET SUMMARY\n{SEP}")
        ts = tearsheet(port_ret, bench_6040, bench_eq)
        print(ts.to_string(float_format=lambda x: f"{x:>10.2f}"))
        print()
        print("Saved: output/tearsheet_summary.csv")

        # Turnover & transaction cost
        print(f"\n{SEP}\n4. TURNOVER & TRANSACTION COST\n{SEP}")
        turnover_series, avg_annual_turnover_pct, total_cost_drag_pct = compute_turnover_stats(
            weights_df, tx_cost_bps=TX_COST_BPS
        )
        print(f"  Avg annual turnover      : {avg_annual_turnover_pct:>8.1f}%")
        print(f"  Total cost drag ({TX_COST_BPS}bps) : {total_cost_drag_pct:>8.2f}% cumulative "
            f"(already deducted in port_ret via the per-rebalance drag term)")

        # Regime transition statistics
        print(f"\n{SEP}\n5. REGIME TRANSITION STATISTICS\n{SEP}")
        n_switches, avg_run_len, dur_by_regime = regime_transition_stats(regimes_df)
        print(f"  Number of regime switches : {n_switches}")
        print(f"  Avg run length            : {avg_run_len} rebalances")
        for regime, dur in dur_by_regime.items():
            print(f"    {regime:<6}: {dur} rebalances avg")

        # Verdict
        print(f"\n{SEP}\n6. VERDICT\n{SEP}")
        print("Regime Shift beats 60/40 static portfolio in almost all metrics in the final tear sheet summary")
        #print(f"  {verdict_line(fold_tearsheets, fold_periods_by_name)}")
        print(f"Overall HMM prediction kappa is {kappa:.3f} and overall accuracy is {overall_acc:.1%} which is reasonably good ")
        print(f"The Regime-Shift survives the transaction friction as the 5 bps penalty per rebalance causes total cost drag of {total_cost_drag_pct:>.2f}% cumulative which is already deducted in results")
        print("Walk forward validation is strictly enforced as while applying the multi fold expanding window strategy the hmm is strictly fitted on past data")
        

    print(f"\n{SEP}\n\U0001F4CA  PERFORMANCE METRICS \n{SEP}")
    print("Refer to performance_tear_sheet.txt for detailed performance report")

    # Save CSVs
    ts.to_csv("output/tearsheet_summary.csv")
    weights_df["regime"] = weights_df["regime"].map({-1: "Bear", 1: "Bull"})
    weights_df.to_csv("output/walk_forward_results.csv")
    weights_df["regime"] = weights_df["regime"].map({"Bear": -1, "Bull": 1})
    print("\nSaved CSVs: tearsheet_summary.csv, walk_forward_results.csv, tearsheet_all_folds.csv")

    # 5. Charts 
    print(f"\n{SEP}\n\U0001F3A8  CHARTS\n{SEP}")
    chart_equity_curves(port_ret, bench_6040, bench_eq, regimes_df)
    chart_drawdown(port_ret, bench_6040, bench_eq)
    chart_rolling_sharpe(port_ret, bench_6040, bench_eq)
    chart_weight_allocation(weights_df)
    chart_regime_frequency(regimes_df)
    chart_monthly_heatmap(port_ret)
    chart_hmm_transitions_grid(fold_models, fold_mappings)

    print(f"\n{SEP}\n\u2705  DONE \u2014 all outputs in ./output/\n{SEP}\n")

    return {
        "prices":      close_px,
        "ohlc":        ohlc,
        "vix":         vix,
        "fred":        fred_df,
        "features":    feat_df,
        "returns":     rets_aligned,
        "port_ret":    port_ret,
        "weights_df":  weights_df,
        "regimes_df":  regimes_df,
        "bench_6040":  bench_6040,
        "bench_eq":    bench_eq,
        "tearsheet":   ts,
        "fold_tearsheets": fold_tearsheets,
        "fold_models": fold_models,
        "fold_mappings": fold_mappings,
        "confusion_matrix": cm_df,
        "kappa": kappa,
        "fold_confusion_matrices": fold_cm_results,
    }


if __name__ == "__main__":
    api_key = os.getenv("FRED_API_KEY", FRED_API_KEY)
    results = run(fred_api_key=api_key)



───────────────────────────────────────────────────────
📥  DATA INGESTION
───────────────────────────────────────────────────────
  ✓ Yahoo Finance prices: 4126 days × 8 assets
  ✓ CBOE VIX: 4127 observations
  ✓ FRED [DGS3MO]: 3-Month Treasury Bill Rate
  ✓ FRED [DGS10]: 10-Year Treasury Bond Rate

───────────────────────────────────────────────────────
📐  FEATURE ENGINEERING 
───────────────────────────────────────────────────────
  Feature matrix: (4124, 9)
  Features: ['ema14', 'daily_range', 'ewma20_vol', 'macd_main', 'vix', 'skew_idx', 'skew_idx_lag', 't3mo', 'yield_ratio']
  Correlation between features: 


,ema14,daily_range,ewma20_vol,macd_main,vix,skew_idx,skew_idx_lag,t3mo,yield_ratio
ema14,1.00,-0.56,-0.29,0.63,-0.48,0.14,0.12,0.03,0.03
daily_range,-0.56,1.00,0.70,-0.58,0.75,-0.20,-0.19,-0.03,-0.03
ewma20_vol,-0.29,0.70,1.00,-0.64,0.87,-0.26,-0.26,-0.03,-0.03
macd_main,0.63,-0.58,-0.64,1.00,-0.60,0.26,0.25,0.06,0.05
vix,-0.48,0.75,0.87,-0.60,1.00,-0.20,-0.19,-0.10,-0.11
skew_idx,0.14,-0.20,-0.26,0.26,-0.20,1.00,0.95,0.43,0.34
skew_idx_lag,0.12,-0.19,-0.26,0.25,-0.19,0.95,1.00,0.43,0.34
t3mo,0.03,-0.03,-0.03,0.06,-0.10,0.43,0.43,1.00,0.95
yield_ratio,0.03,-0.03,-0.03,0.05,-0.11,0.34,0.34,0.95,1.00



───────────────────────────────────────────────────────
🔁  WALK-FORWARD BACKTEST 
───────────────────────────────────────────────────────

Fold time periods used in walk-forward backtester and in results: 
  Fold       Train start  Train end    Test start   Test end    
  Fold 1     2010-01-01   2017-01-01   2017-01-01   2019-01-01  
  Fold 2     2010-01-01   2019-01-01   2019-01-01   2021-01-01  
  Fold 3     2010-01-01   2021-01-01   2021-01-01   2023-01-01  
  Fold 4     2010-01-01   2023-01-01   2023-01-01   2025-01-01  
  Fold 5     2010-01-01   2025-01-01   2025-01-01   2026-05-31  

Refer to walk_forward_backtest.txt for detailed walk forward result which includes details about hmm prediction and portfolio reallocation

────────────────────────────────────────────────────────────
📊  PERFORMANCE METRICS 
────────────────────────────────────────────────────────────
Refer to performance_tear_sheet.txt for detailed performance report

Saved CSVs: tearsheet_summary.csv, walk_forward